# BirthAtlas — Country Comparison Engine (v2.2)

Goal: build a single reusable function that, given a country and a year, returns every
comparison metric needed by later features (Birth Story v2.3, Lottery Simulator v2.4).

Input: `data/processed/birth_probability_hdi.csv` (produced in v2.1).

No visualization here — this notebook is for developer-facing logic only.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/birth_probability_hdi.csv")
df.head()

,Country Name,Country Code,Year,Births,BirthProbability,Rank,HDI
0,Afghanistan,AFG,1993,759621.205448,0.557,32,0.311
1,Albania,ALB,1993,78080.981678,0.057,117,0.624
2,Algeria,DZA,1993,767521.576624,0.563,30,0.603
3,Argentina,ARG,1993,737562.076332,0.541,35,0.748
4,Armenia,ARM,1993,63209.788928,0.046,127,0.621


## Step 1 — Annual HDI rank

`HDI_Rank` must be computed **within each year**, not across the full history,
since the number of countries with available HDI data differs per year.
Rows with missing `HDI` get `NaN` rank rather than being dropped from the dataframe —
we keep them so Birth Probability data for that country/year is not lost.

In [2]:
df["HDI_Rank"] = df.groupby("Year")["HDI"].rank(ascending=False, method="min")

df[["Country Name", "Year", "HDI", "HDI_Rank"]].sort_values(["Year", "HDI_Rank"]).head(10)

,Country Name,Year,HDI,HDI_Rank
135,United States,1993,0.884,1.0
97,Norway,1993,0.875,2.0
5,Australia,1993,0.872,3.0
22,Canada,1993,0.871,4.0
120,Switzerland,1993,0.867,5.0
65,Japan,1993,0.865,6.0
56,Iceland,1993,0.863,7.0
91,Netherlands,1993,0.863,7.0
11,Belgium,1993,0.860,9.0
46,Germany,1993,0.859,10.0


## Step 2 — Global yearly averages

Precompute the global average `BirthProbability` and `HDI` for every year once,
instead of recalculating inside the comparison function on every call.

In [3]:
yearly_avg = (
    df.groupby("Year")
    .agg(
        AvgBirthProbability=("BirthProbability", "mean"),
        AvgHDI=("HDI", "mean"),
        CountriesWithHDI=("HDI", lambda s: s.notna().sum()),
        TotalCountries=("Country Code", "nunique"),
    )
    .reset_index()
)

yearly_avg.head()

,Year,AvgBirthProbability,AvgHDI,CountriesWithHDI,TotalCountries
0,1993,0.656092,0.618127,142,142
1,1994,0.655542,0.622810,142,142
2,1995,0.627711,0.624973,149,149
3,1996,0.626973,0.630638,149,149
4,1997,0.626456,0.635725,149,149


## Step 3 — Core comparison function

`get_country_summary(df, yearly_avg, country_code, year)` returns a dictionary with
every metric needed downstream:

- `BirthProbability`, `BirthRank`
- `HDI`, `HDI_Rank` (may be `None` if missing for that country/year)
- `EstimatedBirths`
- `RelativeToAverage` = BirthProbability / AvgBirthProbability for that year
- `CountriesWithHigherHDI`, `CountriesWithLowerHDI` (only computed when HDI is available)

If the country/year combination does not exist at all, the function raises a clear
`ValueError` rather than silently returning partial/incorrect data.

In [6]:
def get_country_summary(df: pd.DataFrame, yearly_avg: pd.DataFrame, country_code: str, year: int) -> dict:
    """Return a dictionary of comparison metrics for one country in one year.

    Parameters
    ----------
    df : full long-format dataframe including BirthProbability, Rank, HDI, HDI_Rank
    yearly_avg : precomputed per-year global averages (see Step 2)
    country_code : ISO-3 country code, e.g. "IRN"
    year : calendar year, e.g. 2007

    Raises
    ------
    ValueError if no row matches the given country_code/year combination.
    """
    row = df[(df["Country Code"] == country_code) & (df["Year"] == year)]

    if row.empty:
        raise ValueError(
            f"No data found for country_code='{country_code}', year={year}. "
            "Check spelling/ISO code and confirm the year exists in the dataset."
        )

    row = row.iloc[0]
    avg_row = yearly_avg[yearly_avg["Year"] == year].iloc[0]

    has_hdi = pd.notna(row["HDI"])

    summary = {
        "CountryName": row["Country Name"],
        "CountryCode": country_code,
        "Year": year,
        "BirthProbability": round(float(row["BirthProbability"]), 3),
        "BirthRank": int(row["Rank"]),
        "EstimatedBirths": round(float(row["Births"])),
        "RelativeToGlobalAverage": round(float(row["BirthProbability"]) / avg_row["AvgBirthProbability"], 2),
        "HDI": round(float(row["HDI"]), 3) if has_hdi else None,
        "HDI_Rank": int(row["HDI_Rank"]) if has_hdi else None,
        "CountriesWithHigherHDI": None,
        "CountriesWithLowerHDI": None,
        "TotalCountriesThisYear": int(avg_row["TotalCountries"]),
    }

    if has_hdi:
        year_df = df[df["Year"] == year]
        summary["CountriesWithHigherHDI"] = int((year_df["HDI"] < row["HDI"]).sum())
        summary["CountriesWithLowerHDI"] = int((year_df["HDI"] > row["HDI"]).sum())

    return summary

## Step 4 — Quick sanity test (normal case)

In [7]:
get_country_summary(df, yearly_avg, country_code="IRN", year=2007)

{'CountryName': 'Iran, Islamic Rep.',
 'CountryCode': 'IRN',
 'Year': 2007,
 'BirthProbability': 0.914,
 'BirthRank': 21,
 'EstimatedBirths': 1266082,
 'RelativeToGlobalAverage': np.float64(1.74),
 'HDI': 0.761,
 'HDI_Rank': 69,
 'CountriesWithHigherHDI': 118,
 'CountriesWithLowerHDI': 68,
 'TotalCountriesThisYear': 187}

## Step 5 — Edge case discovery: countries missing HDI in some years

This cell scans the dataframe and lists every (country, year) pair where
`BirthProbability` exists but `HDI` is missing — exactly the case the comparison
function must handle gracefully (returning `None` instead of crashing).

In [8]:
missing_hdi = df[df["HDI"].isna()][["Country Name", "Country Code", "Year"]].sort_values(["Country Name", "Year"])

print(f"Total (country, year) rows missing HDI: {len(missing_hdi)}")
print(f"Distinct countries affected: {missing_hdi['Country Code'].nunique()}")
print()
missing_hdi.head(20)

Total (country, year) rows missing HDI: 0
Distinct countries affected: 0



,Country Name,Country Code,Year


## Step 6 — Verify the function on one of the discovered edge cases

Pick the first country/year combination found above and confirm the function
returns `HDI: None` / `HDI_Rank: None` instead of raising an exception.

In [9]:
if len(missing_hdi) > 0:
    sample = missing_hdi.iloc[0]
    result = get_country_summary(df, yearly_avg, country_code=sample["Country Code"], year=int(sample["Year"]))
    print(result)
else:
    print("No missing-HDI rows found in this dataset — edge case does not apply.")

No missing-HDI rows found in this dataset — edge case does not apply.


## Step 7 — Verify behaviour on a fully invalid input (no matching row at all)

This should raise `ValueError`, not return empty/incorrect data.

In [10]:
try:
    get_country_summary(df, yearly_avg, country_code="ZZZ", year=1900)
except ValueError as e:
    print(f"Raised as expected: {e}")

Raised as expected: No data found for country_code='ZZZ', year=1900. Check spelling/ISO code and confirm the year exists in the dataset.
